# PUDO Complaint Risk — Waymo CPUC Deployment Data (2024 Jun – 2026 Q2)

This notebook evaluates whether PUDO complaint frequency changes over time
after controlling for exposure. Run `pudo-build` first. The pipeline validates
carrier identity, legacy/new schemas, month coverage, trips, and VMT before it
writes the Parquet files used here.


In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import statsmodels.api as sm

DATA = Path(os.environ.get("PUDO_DATA_DIR", "../data")) / "parquet"
paths = {
    "summary": DATA / "pudo_summary.parquet",
    "monthly": DATA / "monthly_activity.parquet",
    "complaints": DATA / "complaints.parquet",
}
missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Run pudo-build first; missing: {missing}")

summary = pl.read_parquet(paths["summary"])
monthly = pl.read_parquet(paths["monthly"])
complaints = pl.read_parquet(paths["complaints"])
summary


In [ ]:
summary.select(
    "period_id",
    "period_midpoint",
    "pudo_complaints",
    "pudo_collisions",
    "vmt",
    "trips",
    "pudo_per_100k_vmt",
    "pudo_per_100k_trips",
)


In [ ]:
complaints.group_by(["period_id", "record_grain"]).agg(
    pl.len().alias("source_rows"),
    pl.col("ComplaintPUDO").sum().alias("pudo_complaints"),
    pl.col("PUDOTravelLane").sum().alias("pudo_travel_lane"),
    pl.col("CollisionPUDOAny").sum().alias("pudo_collisions"),
).sort("period_id")


In [ ]:
# Notebook-side guard against stale or manually edited Parquet files.
required = [
    "pudo_complaints",
    "pudo_collisions",
    "vmt",
    "trips",
    "pudo_per_100k_vmt",
    "pudo_per_100k_trips",
    "time_months",
]
null_counts = summary.select(
    [pl.col(column).null_count().alias(column) for column in required]
).row(0)
bad = {column: count for column, count in zip(required, null_counts) if count}
if bad:
    raise ValueError(f"Summary contains null critical values: {bad}")
if summary.filter((pl.col("vmt") <= 0) | (pl.col("trips") <= 0)).height:
    raise ValueError("Summary contains non-positive exposure")
if summary.get_column("period_id").n_unique() != summary.height:
    raise ValueError("Summary contains duplicate reporting periods")

print(f"Validated {summary.height} reporting periods; no critical null values.")


## 1. Raw counts and two exposure-adjusted rates

Raw counts can rise as service grows. We report both complaints per VMT and
complaints per trip: VMT measures road exposure, while trips more directly
measure the number of pickup/drop-off opportunities.


In [ ]:
s = summary.sort("period_midpoint").to_pandas()
labels = s["period_id"]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].bar(labels, s["pudo_complaints"], color="#c44")
axes[0, 0].set_title("PUDO complaints (raw count)")
axes[0, 1].bar(labels, s["vmt"] / 1e6, color="#777")
axes[0, 1].set_title("Exposure: VMT (millions of miles)")
axes[1, 0].plot(labels, s["pudo_per_100k_vmt"], marker="o", color="#26a")
axes[1, 0].set_title("PUDO complaints per 100k VMT")
axes[1, 1].plot(labels, s["pudo_per_100k_trips"], marker="o", color="#287a3b")
axes[1, 1].set_title("PUDO complaints per 100k trips")
for ax in axes.flat:
    ax.tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()


## 2. Trend tests — Poisson regression with exposure offsets

The model uses the actual midpoint of each reporting window. This avoids
treating the one-month December 2024 stub as if it were separated by the same
amount of time as a normal three-month period. Robust standard errors are used.


In [ ]:
def fit_rate_trend(exposure: str):
    columns = ["pudo_complaints", exposure, "time_months"]
    data = s.dropna(subset=columns).copy()
    if len(data) < 4:
        raise ValueError(f"Need at least four valid periods for {exposure}")
    if (data[exposure] <= 0).any():
        raise ValueError(f"{exposure} must be positive")
    if data["pudo_complaints"].sum() <= 0:
        raise ValueError("No PUDO complaints are available for a trend model")

    X = sm.add_constant(data[["time_months"]])
    model = sm.GLM(
        data["pudo_complaints"],
        X,
        family=sm.families.Poisson(),
        offset=np.log(data[exposure]),
    ).fit(cov_type="HC0")

    beta = model.params["time_months"]
    low, high = model.conf_int().loc["time_months"]
    annual_change = np.exp(beta * 12) - 1
    annual_ci = np.exp(np.array([low, high]) * 12) - 1
    dispersion = model.pearson_chi2 / model.df_resid
    direction = "RISING" if annual_change > 0 else "FALLING"
    print(
        f"{exposure}: {direction} {annual_change:+.1%} per year "
        f"(95% CI {annual_ci[0]:+.1%} to {annual_ci[1]:+.1%}), "
        f"p={model.pvalues['time_months']:.4f}, dispersion={dispersion:.2f}"
    )
    if dispersion > 1.5:
        print("  Warning: overdispersion is material; report an NB sensitivity model.")
    return model


vmt_model = fit_rate_trend("vmt")
trip_model = fit_rate_trend("trips")


Interpret the coefficient as a change in the complaint *rate*, not a change in
raw complaint counts. The VMT and trip models are complementary sensitivity
checks. If overdispersion is material, add a negative-binomial model and report
whether its direction and uncertainty agree with the Poisson result.


## 3. Supporting views: travel-lane PUDO and PUDO collisions

These signals must remain separate. Legacy reports may redact travel-lane
totals; gaps in that series mean “unavailable,” not zero.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(labels, s["pudo_per_100k_vmt"], marker="o", label="Complaints / 100k VMT")
ax.plot(
    labels,
    s["pudo_travel_lane"] / s["vmt"] * 1e5,
    marker="s",
    label="Travel-lane PUDO / 100k VMT",
)
ax.plot(
    labels,
    s["pudo_collisions"] / s["vmt"] * 1e5,
    marker="^",
    label="PUDO collisions / 100k VMT",
)
ax.tick_params(axis="x", rotation=60)
ax.set_ylabel("Rate per 100k VMT")
ax.legend()
plt.tight_layout()
plt.show()


## 4. Risk-matrix placement

Do not pre-label this risk as “high likelihood / low consequence.” First define
the course matrix thresholds, then use:

- **Likelihood evidence:** both exposure-adjusted rate series and their trend
  estimates, with uncertainty.
- **Consequence evidence:** complaint categories and PUDO collisions, reported
  separately because a complaint is not equivalent to physical harm.
- **Data limitations:** redaction, changing CPUC schemas, only nine reporting
  windows, and possible changes in complaint-reporting behavior.

The final cell and direction arrow should be filled only after the validated
pipeline is run on all periods and the team agrees on explicit matrix cutoffs.
